# MV-Adapter · NILE-ViewTime Colab 对比实验

这份 notebook 专门运行刚加入项目的 NILE-ViewTime 实验闭环：

1. 将模型缓存、输入、输出和 manifest 持久化到 Google Drive。
2. 对比 IID、Shared、Low-pass Shared、Flat Sobol、NILE-V、NILE-VT、NILE-VTP。
3. 支持 seed、rho、rho_start、active_ratio 网格和断点续跑。
4. 计算 adjacent/opposite 的 lightweight 指标，并可选运行官方 MEt3R。

代码仍由本地 Git 管理；Colab 会从 GitHub fork 克隆最新分支。运行前请先 push 本次新增的 NILE 文件。


## 1. 挂载 Google Drive

checkpoint 下载阶段只需要 CPU。真正开始 SDXL I2MV 推理时再连接 GPU。


In [ ]:
try:
    from google.colab import drive
except ImportError as exc:
    raise RuntimeError(
        "当前 kernel 不是 Google Colab；请在 VS Code 右上角 Select Kernel > Colab 中连接。"
    ) from exc

drive.mount("/content/drive")


: 

## 2. 实验配置

默认 QUICK_MODE=True：只取一个输入、一个 seed 和一个 rho，但仍对比全部七种方法。这样适合先检查代码、显存和输出格式。

完整实验会非常耗时，因为 grid runner 为了可恢复和严格复现，会让每个配置在独立进程中加载一次模型。确认 quick run 正常后再把 QUICK_MODE 改成 False；只在需要 callback 二维超参消融时再开启 SWEEP_CALLBACK_HYPERPARAMS。


In [ ]:
import json
import os
import shlex
import shutil
import subprocess
import sys
from pathlib import Path

GITHUB_REPO = "https://github.com/WANG-Ruipeng/MV-Adapter.git"
BRANCH = "main"
COLAB_REPO = Path("/content/MV-Adapter")

MODEL_ROOT = Path("/content/drive/MyDrive/ModelWeights/MV-Adapter")
PROJECT_DRIVE_ROOT = Path("/content/drive/MyDrive/Colab_Projects/MV-Adapter")
ADAPTER_DIR = MODEL_ROOT / "mv-adapter"
HF_HOME = MODEL_ROOT / "huggingface"

EXPERIMENT_ROOT = PROJECT_DRIVE_ROOT / "nile_viewtime"
INPUT_DIR = EXPERIMENT_ROOT / "inputs"
OUTPUT_ROOT = EXPERIMENT_ROOT / "runs"
PLAN_MANIFEST = EXPERIMENT_ROOT / "plan.json"
MANIFEST_FILE = EXPERIMENT_ROOT / "manifest.json"
EXPERIMENT_CONFIG_FILE = EXPERIMENT_ROOT / "notebook_config.json"
LIGHTWEIGHT_METRICS_FILE = EXPERIMENT_ROOT / "metrics_lightweight.json"
MET3R_METRICS_FILE = EXPERIMENT_ROOT / "metrics_met3r.json"

BASE_MODEL = "stabilityai/stable-diffusion-xl-base-1.0"
VAE_MODEL = "madebyollin/sdxl-vae-fp16-fix"
ADAPTER_REPO = "huanngzh/mv-adapter"
ADAPTER_WEIGHT = "mvadapter_i2mv_sdxl.safetensors"

TEXT_PROMPT = "high quality, detailed object"
AZIMUTHS = [0, 45, 90, 180, 270, 315]
METHODS = [
    "iid",
    "shared",
    "lowpass_shared",
    "flat_sobol",
    "nile_v",
    "nile_vt",
    "nile_vtp",
]

QUICK_MODE = True
SWEEP_CALLBACK_HYPERPARAMS = False
SEEDS = [0] if QUICK_MODE else [0, 1, 2, 3, 4]
RHOS = [0.65] if QUICK_MODE else [0.0, 0.25, 0.5, 0.65, 0.8, 1.0]
RHO_STARTS = (
    [0.25, 0.45, 0.65]
    if SWEEP_CALLBACK_HYPERPARAMS and not QUICK_MODE
    else [0.45]
)
ACTIVE_RATIOS = (
    [0.4, 0.6, 0.8]
    if SWEEP_CALLBACK_HYPERPARAMS and not QUICK_MODE
    else [0.6]
)
NUM_INFERENCE_STEPS = 30 if QUICK_MODE else 50
REMOVE_BACKGROUND = True

for path in (
    ADAPTER_DIR,
    HF_HOME,
    EXPERIMENT_ROOT,
    INPUT_DIR,
    OUTPUT_ROOT,
):
    path.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["PYTHONUNBUFFERED"] = "1"

print("模式:", "QUICK" if QUICK_MODE else "FULL")
print("Callback 超参 sweep:", SWEEP_CALLBACK_HYPERPARAMS)
print("临时代码:", COLAB_REPO)
print("Adapter:", ADAPTER_DIR)
print("HF cache:", HF_HOME)
print("实验目录:", EXPERIMENT_ROOT)
print("预计每个输入的配置数会在 dry-run cell 中给出。")


## 3. 下载 checkpoint 到 Drive（CPU 即可）

MV-Adapter 权重会直接保存到 Drive。基础 SDXL 和 VAE 默认在第一次推理时进入同一个 Drive Hugging Face cache。

如果希望在 CPU runtime 中提前把基础模型也下载好，把 PRELOAD_MODEL_SNAPSHOTS 改成 True。上述仓库通常可公开访问；若你的模型需要授权，可把 token 放进 Colab Secrets 的 HF_TOKEN，或开启安全输入框。


In [ ]:
from getpass import getpass

try:
    from huggingface_hub import hf_hub_download, snapshot_download
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"],
        check=True,
    )
    from huggingface_hub import hf_hub_download, snapshot_download

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

PROMPT_FOR_HF_TOKEN = False
if PROMPT_FOR_HF_TOKEN and not hf_token:
    hf_token = getpass("Hugging Face token（输入不会显示，留空表示匿名）: ").strip() or None

if hf_token:
    os.environ["HF_TOKEN"] = hf_token

adapter_file = Path(
    hf_hub_download(
        repo_id=ADAPTER_REPO,
        filename=ADAPTER_WEIGHT,
        local_dir=ADAPTER_DIR,
        token=hf_token,
    )
)
print(f"Adapter ready: {adapter_file} ({adapter_file.stat().st_size / 1024**2:.1f} MiB)")

PRELOAD_MODEL_SNAPSHOTS = False
if PRELOAD_MODEL_SNAPSHOTS:
    for repo_id in (BASE_MODEL, VAE_MODEL):
        print("Downloading:", repo_id)
        snapshot_download(
            repo_id=repo_id,
            cache_dir=HF_HOME / "hub",
            token=hf_token,
        )
    print("基础模型和 VAE 已缓存到 Drive。")
else:
    print("跳过基础模型预下载；第一次推理时会自动写入 HF cache。")


## 4. 检查 Colab GPU

SDXL I2MV 通常需要约 14GB 显存。优先选择 A100；L4/A10G 一般也可以。T4 只有较小余量，若出现 OOM，先关闭背景移除、减少同时运行的其他进程，或更换 GPU。


In [ ]:
import torch

assert torch.cuda.is_available(), "没有检测到 CUDA GPU；请切换到带 GPU 的 Colab runtime。"
subprocess.run(["nvidia-smi"], check=True)

device_name = torch.cuda.get_device_name(0)
free_bytes, total_bytes = torch.cuda.mem_get_info()
print("PyTorch:", torch.__version__)
print("GPU:", device_name)
print(f"显存: free={free_bytes / 1024**3:.1f} GiB / total={total_bytes / 1024**3:.1f} GiB")
if total_bytes < 15 * 1024**3:
    print("警告：显存余量可能不足；建议使用 L4/A10G/A100。")


## 5. 克隆或更新 GitHub 代码

本地 workspace 不会自动出现在 Colab 文件系统中。下面会拉取 fork 的指定分支，并明确检查新实验脚本是否已经 push。


In [ ]:
if (COLAB_REPO / ".git").is_dir():
    subprocess.run(["git", "-C", str(COLAB_REPO), "checkout", BRANCH], check=True)
    subprocess.run(
        ["git", "-C", str(COLAB_REPO), "pull", "--ff-only", "origin", BRANCH],
        check=True,
    )
elif COLAB_REPO.exists():
    raise RuntimeError(f"{COLAB_REPO} 已存在但不是 Git 仓库，请更换路径或手动清理。")
else:
    subprocess.run(
        [
            "git",
            "clone",
            "--depth",
            "1",
            "--branch",
            BRANCH,
            GITHUB_REPO,
            str(COLAB_REPO),
        ],
        check=True,
    )

required_files = [
    COLAB_REPO / "scripts/inference_i2mv_sdxl_nile.py",
    COLAB_REPO / "scripts/run_nile_grid.py",
    COLAB_REPO / "scripts/eval_multiview_consistency.py",
    COLAB_REPO / "mvadapter/nile/sampler.py",
]
missing = [str(path.relative_to(COLAB_REPO)) for path in required_files if not path.is_file()]
assert not missing, (
    "GitHub 分支还没有本次 NILE 更新，请先在本地 commit/push。缺少: "
    + ", ".join(missing)
)

subprocess.run(["git", "-C", str(COLAB_REPO), "log", "-1", "--oneline"], check=True)


## 6. 安装 Colab 推理依赖

安装日志会完整输出；若失败，不会只显示一个 CalledProcessError。nvdiffrast 单独安装，避免 build isolation 找不到 Colab 已有的 PyTorch。


In [ ]:
from pathlib import Path

def run_pip(*args):
    command = [sys.executable, "-m", "pip", *map(str, args)]
    print("\n$", shlex.join(command), flush=True)
    result = subprocess.run(
        command,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
    )
    print(result.stdout, flush=True)
    result.check_returncode()

requirements_file = COLAB_REPO / "requirements-colab.txt"
assert requirements_file.is_file(), requirements_file

run_pip("install", "-r", requirements_file)
run_pip("install", "setuptools", "wheel", "ninja")
run_pip(
    "install",
    "git+https://github.com/NVlabs/nvdiffrast.git",
    "--no-build-isolation",
)
run_pip("install", "-e", COLAB_REPO, "--no-deps")

def run_streaming(command, *, cwd=COLAB_REPO, env=None):
    command = [str(item) for item in command]
    print("\n$", shlex.join(command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=str(cwd),
        env=env or os.environ.copy(),
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise RuntimeError(
            f"命令失败，退出码 {return_code}: {shlex.join(command)}"
        )

print("依赖和 MV-Adapter 已安装。")


## 7. 准备实验输入

把自己的 PNG/JPG/WebP 放到 Drive 的 nile_viewtime/inputs 目录。如果目录为空，下面会自动复制仓库中的企鹅示例，确保 notebook 可以直接跑通。

QUICK_MODE 只使用排序后的第一张图；FULL 模式使用目录中的全部图片。


In [ ]:
SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".webp", ".bmp"}

def discover_inputs():
    return sorted(
        path.resolve()
        for path in INPUT_DIR.iterdir()
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )

input_images = discover_inputs()
if not input_images:
    demo_source = (
        COLAB_REPO
        / "assets/demo/i2mv/A_juvenile_emperor_penguin_chick.png"
    )
    assert demo_source.is_file(), demo_source
    demo_target = INPUT_DIR / demo_source.name
    shutil.copy2(demo_source, demo_target)
    print("已复制默认输入:", demo_target)
    input_images = discover_inputs()

INPUT_IMAGES = input_images[:1] if QUICK_MODE else input_images
assert INPUT_IMAGES, f"{INPUT_DIR} 中没有可用图片"
print("本轮输入:")
for path in INPUT_IMAGES:
    print(" -", path)


## 8. 可选：单方法冒烟测试

这个 cell 会额外加载一次 SDXL，因此默认关闭。遇到 grid 首任务失败时，可以开启 RUN_SMOKE_TEST，先用 NILE-V 单独定位问题。


In [ ]:
from IPython.display import Image as IPyImage
from IPython.display import display

RUN_SMOKE_TEST = False
SMOKE_OUTPUT = EXPERIMENT_ROOT / "smoke_nile_v.png"

if RUN_SMOKE_TEST:
    smoke_command = [
        sys.executable,
        "-m",
        "scripts.inference_i2mv_sdxl_nile",
        "--base_model",
        BASE_MODEL,
        "--vae_model",
        VAE_MODEL,
        "--adapter_path",
        str(ADAPTER_DIR),
        "--image",
        str(INPUT_IMAGES[0]),
        "--text",
        TEXT_PROMPT,
        "--seed",
        "0",
        "--num_inference_steps",
        "20",
        "--guidance_scale",
        "3.0",
        "--azimuth_deg",
        *map(str, AZIMUTHS),
        "--nile_mode",
        "nile_v",
        "--nile_callback",
        "none",
        "--rho_geo",
        "0.65",
        "--output",
        str(SMOKE_OUTPUT),
    ]
    if REMOVE_BACKGROUND:
        smoke_command.append("--remove_bg")
    run_streaming(smoke_command)
    display(IPyImage(filename=str(SMOKE_OUTPUT)))
else:
    print("已跳过。需要时把 RUN_SMOKE_TEST 改成 True。")


## 9. Dry-run：检查实验矩阵

只生成计划和命令，不加载模型。默认会自动去掉 IID、Shared、Flat Sobol 在不同 rho 下的无意义重复。


In [ ]:
def build_grid_command(*, dry_run):
    manifest_path = PLAN_MANIFEST if dry_run else MANIFEST_FILE
    command = [
        sys.executable,
        "-m",
        "scripts.run_nile_grid",
        "--input",
        *map(str, INPUT_IMAGES),
        "--methods",
        *METHODS,
        "--seeds",
        *map(str, SEEDS),
        "--rhos",
        *map(str, RHOS),
        "--rho-starts",
        *map(str, RHO_STARTS),
        "--active-ratios",
        *map(str, ACTIVE_RATIOS),
        "--rho-end",
        "0.0",
        "--text",
        TEXT_PROMPT,
        "--azimuth-deg",
        *map(str, AZIMUTHS),
        "--num-inference-steps",
        str(NUM_INFERENCE_STEPS),
        "--guidance-scale",
        "3.0",
        "--base-model",
        BASE_MODEL,
        "--vae-model",
        VAE_MODEL,
        "--adapter-path",
        str(ADAPTER_DIR),
        "--device",
        "cuda",
        "--output-root",
        str(OUTPUT_ROOT),
        "--manifest",
        str(manifest_path),
    ]
    if REMOVE_BACKGROUND:
        command.append("--remove-bg")
    command.append("--dry-run" if dry_run else "--resume")
    return command

import importlib.metadata as package_metadata

def installed_version(name):
    try:
        return package_metadata.version(name)
    except package_metadata.PackageNotFoundError:
        return None

git_commit = subprocess.check_output(
    ["git", "-C", str(COLAB_REPO), "rev-parse", "HEAD"],
    text=True,
).strip()
adapter_checkpoint = ADAPTER_DIR / ADAPTER_WEIGHT
experiment_config = {
    "git_commit": git_commit,
    "github_repo": GITHUB_REPO,
    "branch": BRANCH,
    "base_model": BASE_MODEL,
    "vae_model": VAE_MODEL,
    "adapter_repo": ADAPTER_REPO,
    "adapter_weight": ADAPTER_WEIGHT,
    "adapter_size_bytes": adapter_checkpoint.stat().st_size,
    "gpu": torch.cuda.get_device_name(0),
    "packages": {
        name: installed_version(name)
        for name in (
            "torch",
            "diffusers",
            "transformers",
            "accelerate",
            "huggingface-hub",
            "mvadapter",
        )
    },
    "inputs": [str(path) for path in INPUT_IMAGES],
    "text": TEXT_PROMPT,
    "azimuths": AZIMUTHS,
    "methods": METHODS,
    "seeds": SEEDS,
    "rhos": RHOS,
    "rho_starts": RHO_STARTS,
    "active_ratios": ACTIVE_RATIOS,
    "num_inference_steps": NUM_INFERENCE_STEPS,
    "remove_background": REMOVE_BACKGROUND,
}
EXPERIMENT_CONFIG_FILE.write_text(
    json.dumps(experiment_config, indent=2, ensure_ascii=False) + "\n",
    encoding="utf-8",
)
print("实验配置快照:", EXPERIMENT_CONFIG_FILE)

run_streaming(build_grid_command(dry_run=True))

plan_payload = json.loads(PLAN_MANIFEST.read_text(encoding="utf-8"))
plan_records = plan_payload["runs"]
print(f"\n计划任务数: {len(plan_records)}")
for method in METHODS:
    count = sum(record["method"] == method for record in plan_records)
    print(f"{method:16s}: {count}")


## 10. 正式运行对比实验

结果、逐视图 PNG、metadata 和 manifest 全部写入 Drive。中断或 runtime 重启后重新运行此 cell，--resume 会跳过已经完整生成的配置。


In [ ]:
RUN_EXPERIMENT = True

if RUN_EXPERIMENT:
    run_streaming(build_grid_command(dry_run=False))
    print("实验 manifest:", MANIFEST_FILE)
else:
    print("已跳过。确认 dry-run 计划后把 RUN_EXPERIMENT 改成 True。")


## 11. 查看运行状态

成功任务需要同时有输出图和 metadata。状态表可帮助发现某个方法或某组参数是否失败。


In [ ]:
import pandas as pd

manifest_payload = json.loads(MANIFEST_FILE.read_text(encoding="utf-8"))
manifest_records = manifest_payload["runs"]
manifest_df = pd.DataFrame(manifest_records)

status_summary = (
    manifest_df.groupby(["method", "status"], dropna=False)
    .size()
    .rename("count")
    .reset_index()
)
display(status_summary)

failed = manifest_df[manifest_df["status"] == "failed"]
if not failed.empty:
    display(
        failed[
            ["method", "seed", "rho_geo", "rho_start", "active_ratio", "error"]
        ]
    )
else:
    print("当前 manifest 没有失败任务。")


## 12. Lightweight adjacent/opposite 评估

这一步不需要 MEt3R 的重型 CUDA 依赖，会计算低频一致性、高频保留、边缘、颜色和全局 SSIM proxy。它们是诊断指标，不是几何感知 MEt3R 的替代品。


In [ ]:
completed_statuses = {"succeeded", "skipped"}
completed_records = [
    record for record in manifest_records
    if record.get("status") in completed_statuses
]
assert completed_records, "manifest 中没有可评估的成功输出"

lightweight_command = [
    sys.executable,
    "-m",
    "scripts.eval_multiview_consistency",
    "--manifest",
    str(MANIFEST_FILE),
    "--metrics",
    "lightweight",
    "--image-size",
    "256",
    "--output",
    str(LIGHTWEIGHT_METRICS_FILE),
]
run_streaming(lightweight_command)
print("评估结果:", LIGHTWEIGHT_METRICS_FILE)


## 13. 汇总对比结果并展示生成图

方向提示：lowfreq_l1_similarity、edge_cosine、color_hist_intersection 越高越好；lowfreq_rmse 和 highfreq_l1_distance 是距离。判断过耦合时不要只看低频一致性，还要同时观察高频多样性和生成图。


In [ ]:
metrics_payload = json.loads(
    LIGHTWEIGHT_METRICS_FILE.read_text(encoding="utf-8")
)
aggregate_df = pd.DataFrame(metrics_payload["aggregates"])

preferred_columns = [
    "method",
    "nile_mode",
    "nile_callback",
    "rho_geo",
    "sample_count",
    "adjacent_lowfreq_l1_similarity",
    "opposite_lowfreq_l1_similarity",
    "adjacent_lowfreq_rmse",
    "opposite_lowfreq_rmse",
    "adjacent_highfreq_l1_distance",
    "opposite_highfreq_l1_distance",
]
visible_columns = [
    column for column in preferred_columns if column in aggregate_df.columns
]
display(
    aggregate_df[visible_columns]
    .sort_values(["method", "rho_geo"], na_position="last")
    .reset_index(drop=True)
)

shown_methods = set()
for record in manifest_records:
    if record.get("status") not in completed_statuses:
        continue
    method = record.get("method")
    output_path = Path(record["output"])
    if method in shown_methods or not output_path.is_file():
        continue
    shown_methods.add(method)
    print(
        f"{method} | seed={record.get('seed')} | "
        f"rho={record.get('rho_geo')} | {output_path}"
    )
    display(IPyImage(filename=str(output_path)))


## 14. 可选：安装并运行官方 MEt3R

MEt3R 依赖 CUDA、PyTorch3D、FeatUp 和 MASt3R/DUSt3R，安装明显比 lightweight 评估重。建议先完成 quick comparison，再单独开启。

默认 cosine MEt3R 定义为 1-cosine 距离，范围 [0, 2]，越低越好。评估 JSON/CSV 会记录 score_direction，避免把方向读反。


In [ ]:
INSTALL_MET3R = False

if INSTALL_MET3R:
    run_pip("install", "git+https://github.com/mohammadasim98/met3r")
    print("MEt3R 安装完成。")
else:
    print("已跳过安装。需要时把 INSTALL_MET3R 改成 True。")


In [ ]:
RUN_MET3R = False

if RUN_MET3R:
    met3r_command = [
        sys.executable,
        "-m",
        "scripts.eval_multiview_consistency",
        "--manifest",
        str(MANIFEST_FILE),
        "--metrics",
        "met3r",
        "--met3r-device",
        "cuda",
        "--met3r-image-size",
        "256",
        "--met3r-batch-size",
        "1",
        "--output",
        str(MET3R_METRICS_FILE),
    ]
    run_streaming(met3r_command)
    met3r_payload = json.loads(
        MET3R_METRICS_FILE.read_text(encoding="utf-8")
    )
    display(pd.DataFrame(met3r_payload["aggregates"]))
else:
    print("已跳过 MEt3R。先安装后把 RUN_MET3R 改成 True。")


## 15. 从 quick run 扩展到完整实验

1. 把 QUICK_MODE 改为 False，重新运行配置 cell。
2. 默认先固定 rho_start=0.45、active_ratio=0.6；需要二维 callback 消融时再开启 SWEEP_CALLBACK_HYPERPARAMS。
3. 在 inputs 目录放入正式输入图；建议先少量运行，再逐步扩展到 20–50 张。
4. 重新运行“准备实验输入”到“正式运行”几个 cell。
5. 保持同一个 OUTPUT_ROOT 和 MANIFEST_FILE；run ID 包含完整配置，--resume 会继续缺失任务。
6. 先用 lightweight 指标排查失败和过耦合，再在筛选后的结果上运行 MEt3R。

研究解释边界：

- 当前实现是 Sobol-backed prototype，不是严格的 NILE/SZ。
- prompt 公式中 lowpass_shared/NILE 的 rho_geo=0 是高通 child noise，不等于 IID；必须保留显式 iid baseline。
- NILE-VTP 当前实现的是 prompt 第一版的 patch Morton 调制，不能宣称已经完成严格的 time-view-patch SZ 层级。
